In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [64]:
df = pd.read_csv("../data/raw/dataset_transacciones/hey_clientes.csv")

In [65]:
df.head(5)

,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,...,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,...,527,1,app_android,10.0,False,True,es_MX,False,2,False
1,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,...,714,3,app_android,8.0,False,True,es_MX,True,2,False
2,USR-00003,23,H,Chihuahua,Cuauhtémoc,Licenciatura,Estudiante,14000,1174,True,...,454,3,app_ios,8.0,False,True,es_MX,False,2,False
3,USR-00004,32,SE,Nuevo León,Guadalupe,Posgrado,Empleado,61000,1168,False,...,837,16,app_ios,10.0,True,False,es_MX,True,3,False
4,USR-00005,26,M,Ciudad de México,CDMX - Cuauhtémoc,Preparatoria,Empresario,27000,816,True,...,533,1,app_ios,7.0,False,True,es_MX,True,2,False


In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15025 entries, 0 to 15024
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  15025 non-null  str    
 1   edad                     15025 non-null  int64  
 2   sexo                     15025 non-null  str    
 3   estado                   14593 non-null  str    
 4   ciudad                   14593 non-null  str    
 5   nivel_educativo          15025 non-null  str    
 6   ocupacion                15025 non-null  str    
 7   ingreso_mensual_mxn      15025 non-null  int64  
 8   antiguedad_dias          15025 non-null  int64  
 9   es_hey_pro               15025 non-null  bool   
 10  nomina_domiciliada       15025 non-null  bool   
 11  canal_apertura           15025 non-null  str    
 12  score_buro               15025 non-null  int64  
 13  dias_desde_ultimo_login  15025 non-null  int64  
 14  preferencia_canal        15025 no

In [67]:
df.isnull().sum()

user_id                      0
edad                         0
sexo                         0
estado                     432
ciudad                     432
nivel_educativo              0
ocupacion                    0
ingreso_mensual_mxn          0
antiguedad_dias              0
es_hey_pro                   0
nomina_domiciliada           0
canal_apertura               0
score_buro                   0
dias_desde_ultimo_login      0
preferencia_canal            0
satisfaccion_1_10          751
recibe_remesas               0
usa_hey_shop                 0
idioma_preferido             0
tiene_seguro                 0
num_productos_activos        0
patron_uso_atipico           0
dtype: int64

In [68]:
df_with_nulls = df[df['estado'].isnull()]

# Mira las primeras 10 columnas de las primeras 5 filas con nulos
print(df_with_nulls.iloc[:5, :10])

      user_id  edad sexo estado ciudad nivel_educativo      ocupacion  \
7   USR-00008    31    H    NaN    NaN    Licenciatura       Empleado   
28  USR-00029    54    H    NaN    NaN    Licenciatura  Independiente   
37  USR-00038    47    H    NaN    NaN    Licenciatura       Empleado   
56  USR-00057    56    H    NaN    NaN    Licenciatura       Empleado   
65  USR-00066    34    H    NaN    NaN      Secundaria       Empleado   

    ingreso_mensual_mxn  antiguedad_dias  es_hey_pro  
7                 11500             1117        True  
28                85000              332       False  
37                34500             1229        True  
56                29000             1012       False  
65                17000              429       False  


In [69]:
df_with_0 = df[df['ciudad'] == "0"]

# Mira las primeras 10 columnas de las primeras 5 filas con nulos
print(df_with_0.iloc[:5, :10])

Empty DataFrame
Columns: [user_id, edad, sexo, estado, ciudad, nivel_educativo, ocupacion, ingreso_mensual_mxn, antiguedad_dias, es_hey_pro]
Index: []


In [70]:
clientes_limpios = df.copy()

columnas_texto = clientes_limpios.select_dtypes(include=["object"]).columns

for col in columnas_texto:
    clientes_limpios[col] = clientes_limpios[col].astype(str).str.strip()

# Corregir los "nan" que se pudieron convertir a texto
clientes_limpios = clientes_limpios.replace("nan", np.nan)

# Rellenar estado y ciudad
clientes_limpios["estado"] = clientes_limpios["estado"].fillna("sin_estado")
clientes_limpios["ciudad"] = clientes_limpios["ciudad"].fillna("sin_ciudad")

# Rellenar satisfacción con la mediana
mediana_satisfaccion = clientes_limpios["satisfaccion_1_10"].median()
clientes_limpios["satisfaccion_1_10"] = clientes_limpios["satisfaccion_1_10"].fillna(mediana_satisfaccion)

# Asegurar tipos numéricos
clientes_limpios["edad"] = pd.to_numeric(clientes_limpios["edad"], errors="coerce")
clientes_limpios["ingreso_mensual_mxn"] = pd.to_numeric(clientes_limpios["ingreso_mensual_mxn"], errors="coerce")
clientes_limpios["antiguedad_dias"] = pd.to_numeric(clientes_limpios["antiguedad_dias"], errors="coerce")
clientes_limpios["score_buro"] = pd.to_numeric(clientes_limpios["score_buro"], errors="coerce")
clientes_limpios["dias_desde_ultimo_login"] = pd.to_numeric(clientes_limpios["dias_desde_ultimo_login"], errors="coerce")
clientes_limpios["num_productos_activos"] = pd.to_numeric(clientes_limpios["num_productos_activos"], errors="coerce")

# Si algo numérico quedó nulo, rellenar con mediana de su columna
columnas_numericas = clientes_limpios.select_dtypes(include=["number"]).columns

for col in columnas_numericas:
    clientes_limpios[col] = clientes_limpios[col].fillna(clientes_limpios[col].median())

C:\Users\CommodorePlus\AppData\Local\Temp\ipykernel_32720\3404103240.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = clientes_limpios.select_dtypes(include=["object"]).columns


In [71]:
print("Filas:", clientes_limpios.shape[0])
print("Columnas:", clientes_limpios.shape[1])
print("Nulos totales:", clientes_limpios.isnull().sum().sum())
print("User_id nulos:", clientes_limpios["user_id"].isnull().sum())
print("User_id duplicados:", clientes_limpios["user_id"].duplicated().sum())

clientes_limpios.isnull().sum().sort_values(ascending=False).head(10)

Filas: 15025
Columnas: 22
Nulos totales: 0
User_id nulos: 0
User_id duplicados: 0


user_id                0
edad                   0
sexo                   0
estado                 0
ciudad                 0
nivel_educativo        0
ocupacion              0
ingreso_mensual_mxn    0
antiguedad_dias        0
es_hey_pro             0
dtype: int64

In [ ]:

ruta_salida = "../data/clean/complete/hey_clientes_limpio.csv"

clientes_limpios.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"✅ Proceso completado. Archivo guardado en: {ruta_salida}")

✅ Proceso completado. Archivo guardado en: ../data/clean/hey_clientes_limpio.csv
